This notebook demonstrates Poisson reduced-rank regression (p-RRR) on a synthetic dataset. It should take less than 5 mins to run the entire notebook.

## Imports

In [ ]:
import os
import sys 
import numpy as np
import matplotlib.pyplot as plt
import time
from mpl_toolkits.axes_grid1 import make_axes_locatable
import sklearn.metrics
from tabulate import tabulate

sys.path.append('../poissonrrr')

from poisson_rrr import *
from linear_rrr import *

## Create synthetic dataset

We build a synthetic count dataset from a known low-rank Poisson model so we can later check whether Poisson reduced-rank regression recovers the structure we planted in the data.

The setup is

$Y \sim \mathrm{Poisson}(\lambda), \qquad \lambda = f(XW + b)$

with the softplus nonlinearity

$f(z) = \log(1 + e^z)$

The role of each term is:

- $X \in \mathbb{R}^{X_0 \times X_1}$: the input design matrix.
- $W \in \mathbb{R}^{X_1 \times Y_1}$: the weight matrix that maps inputs into output-specific linear predictors.
- $b \in \mathbb{R}^{Y_1}$: an output-specific bias term.
- $R$: the target rank of $W$, with $R \leq \min(X_1, Y_1)$.

The low-rank constraint is the important part. Instead of allowing every output to vary independently, we force the outputs to depend on a smaller number of shared factors.

In the code below, `X0`, `X1`, and `Y1` are dimension variables for the number of samples, input features, and output features, respectively.

In [ ]:
# Set a random seed for reproducibility.
seed = 1
rng = np.random.default_rng(seed=seed)

# Define dataset dimensions.
X0 = int(1e4)   # number of samples
X1 = 100        # number of input features
Y1 = 50         # number of output features
R = 10          # target rank of the weight matrix

# Generate synthetic inputs and an unconstrained weight matrix.
X = rng.normal(size=(X0, X1))
W = rng.normal(size=(X1, Y1))

# Compute the SVD once, then truncate it to impose the desired rank.
u, s, vh = np.linalg.svd(W, full_matrices=False)
print(f"shape of SVD components: U: {u.shape}, S: {s.shape}, V^H: {vh.shape}")

W = u[:, :R] @ np.diag(s[:R]) @ vh[:R]
print(W.shape, np.linalg.matrix_rank(W))

The random matrix `W` starts out full-rank. To create a controlled low-rank problem, we decompose it with the singular value decomposition and keep only the top `R` singular directions.

This gives us a new weight matrix with rank `R`. In other words, we are manufacturing data with the kind of shared low-dimensional structure that p-RRR is designed to learn.

In [ ]:
# Draw an output bias term, then convert the linear predictor into mean rates.
b = rng.normal(size=Y1)
print(f"shape of output bias: {b.shape}")

M = X @ W + b
Y_mean = F.softplus(torch.from_numpy(M).float())

# Sample observed counts from the Poisson mean rates.
Y = rng.poisson(Y_mean.numpy())
print(f"shape of observed counts: {Y.shape}")

## Train/test split

Before fitting the model, we partition the synthetic dataset into a training set and a held-out test set.
Here we use an 80/20 split.

In [ ]:
train_fraction = 0.8
split_idx = int(X0 * train_fraction)

X_train, Y_train = X[:split_idx], Y[:split_idx]
X_test, Y_test = X[split_idx:], Y[split_idx:]

print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

## Model fitting

Now we instantiate a Poisson reduced-rank regression model and fit it on the training set.

At a high level, p-RRR combines two ideas:

- Poisson regression, which is appropriate for count-valued responses (like neuron spiking).
- Reduced-rank structure, which constrains the mapping from inputs to outputs to lie in a lower-dimensional subspace.

In this demo, the model rank is set to 10 to match the rank used when generating the synthetic data. That makes this section a recovery experiment: we are checking whether the model can recover the planted low-rank signal when the rank is specified correctly.

In [ ]:
# Match the model rank to the synthetic ground truth.
model_rank = 10

pRRR = PoissonRRR(
    n_input=X1,
    n_output=Y1,
    act="softplus",
    rank=model_rank,
    loss="poisson",
    regList=[0],
    zeromeanXs=[0],
    normstdXs=[0],
    seed=seed,
)

### Training setup

Most of the remaining arguments control the optimizer rather than the statistical model itself. In particular, this implementation uses L-BFGS, a second-order optimization method that often converges quickly for smooth objectives like this one.

In [ ]:
# Fit the model on the training set and keep the optimization history for inspection.
start = time.time()
train_loss_hist, grad_hist = pRRR.train(
    [X_train],
    Y_train,
    lr=1,
    maxepoch=50,
    max_iter=20,
    history_size=10,
    line_search=1,
    grad_tol=1e-4,
    loss_tol=1e-6,
    patience=10,
    shuffle=True,
    track=1,
    progfile=None,
)
elapsed_time = time.time() - start
print(f"elapsed {elapsed_time:.2f}s")

In [ ]:
# Plot the training objective to confirm that optimization stabilizes.
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(train_loss_hist, "-o")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
plt.show()

The training-loss curve should drop quickly because Poisson-RRR here is optimized with L-BFGS, which is typically much faster than plain gradient descent for this problem.

## Test set prediction

After training, evaluate on held-out data to check generalization.

Metrics used below: PNLL (Poisson negative log-likelihood, lower is better), D2 (Poisson deviance explained, higher is better), and R2 (coefficient of determination, higher is better).

In [ ]:
# Evaluate predictive performance on held-out data.
test_scores, Y_test_pred = pRRR.eval(
    [X_test],
    Y_test,
    metrics=["PNLL", "cc", "r2", "d2"],
)

In [ ]:
# Compare the true test-set mean rates against the model predictions.
true_test_mean = Y_mean.numpy()[split_idx:]

fig, (ax_true, ax_pred) = plt.subplots(1, 2, figsize=(4, 4))
ax_true.matshow(true_test_mean, aspect="auto")
ax_true.axis("off")
ax_true.set_title("true")

ax_pred.matshow(Y_test_pred, aspect="auto")
ax_pred.axis("off")
ax_pred.set_title("p-RRR prediction")

fig.suptitle(r"test-set mean $\lambda$")
plt.tight_layout()
plt.show()

## Parameter recovery

Does the fitted model recover the planted weight matrix and bias term?

The next cells extract the fitted parameters from the trained network and compare them against the ground truth.

In [ ]:
# Reconstruct the effective linear map and bias from the fitted two-layer model.
state_dict = pRRR.glm.state_dict()
linear1_weight = state_dict["linear1.weight"].detach().numpy()
linear1_bias = state_dict["linear1.bias"].detach().numpy()
linear2_weight = state_dict["linear2.weight"].detach().numpy()
linear2_bias = state_dict["linear2.bias"].detach().numpy()

W_hat = (linear2_weight @ linear1_weight).T
print(W_hat.shape)

# b_hat is column-shaped (Y1, 1); we flatten it only when plotting.
b_hat = linear2_weight @ linear1_bias.reshape(-1, 1) + linear2_bias.reshape(-1, 1)
print(b_hat.shape)

In [ ]:
# Visualize the recovered weight matrix and bias against the ground truth.
fig, axes = plt.subplot_mosaic(
    [["top left", "top right"], ["bottom row", "bottom row"]],
    figsize=(6, 4),
)

fig.text(0.5, 1, r"weight $W$", ha="center", va="center", fontsize="large")
axes["top left"].matshow(W.T, aspect="auto")
axes["top left"].axis("off")
axes["top left"].set_title("true")

axes["top right"].matshow(W_hat.T, aspect="auto")
axes["top right"].axis("off")
axes["top right"].set_title("p-RRR fit")

axes["bottom row"].bar(
    x=np.arange(Y1),
    height=b_hat.ravel(),
    color="green",
    label="p-RRR fit",
)
axes["bottom row"].bar(
    x=np.arange(Y1),
    height=b,
    color="None",
    edgecolor="orange",
    label="true",
)
axes["bottom row"].legend(ncol=2)
axes["bottom row"].set_title(r"bias $b$")

plt.tight_layout()
plt.show()

## Latent factor recovery (rank-10 fit)

Compare the leading latent factors from the true weight matrix and the fitted p-RRR model (with sign alignment per component).

In [ ]:
# SVDs of true and recovered weight matrices.
u_true, s_true, _ = np.linalg.svd(W, full_matrices=False)
u_hat, s_hat, _ = np.linalg.svd(W_hat, full_matrices=False)

n_factors = R

In [ ]:
fig, axes = plt.subplots(n_factors, 1, sharex=True, figsize=(9, 8))
for i in range(n_factors):
    cosine = np.dot(u_true[:, i], u_hat[:, i]) / (
        np.linalg.norm(u_true[:, i]) * np.linalg.norm(u_hat[:, i])
    )
    aligned_u_hat = u_hat[:, i] if cosine > 0 else -u_hat[:, i]

    axes[i].plot(
        aligned_u_hat * s_hat[i],
        "-o",
        markersize=4,
        color="green",
        label="p-RRR fit",
    )
    axes[i].plot(
        u_true[:, i] * s_true[i],
        "--o",
        markersize=4,
        color="orange",
        markerfacecolor="None",
        label="true",
    )

axes[0].legend(bbox_to_anchor=(1.01, 1.0), loc="upper left")
axes[0].set_title("Latent factors")
plt.show()

## Rank selection

In [ ]:
ranks = [1, 2, 4, 6, 8, 10, 14, 18, 22, 26, 30, 40, 50]
regs = [0, 1e-6, 1e-5, 1e-4, 1e-3]

Run a 2-fold split on the training set to select rank and L2 regularization for p-RRR (~3 minutes).

In [ ]:
half_idx = X_train.shape[0] // 2
X_train1, Y_train1 = X_train[:half_idx], Y_train[:half_idx]
X_val1, Y_val1 = X_train[half_idx:], Y_train[half_idx:]

In [ ]:
PNLL_cv_2fold = np.zeros((2, len(ranks), len(regs)))
d2_cv_2fold = np.zeros((2, len(ranks), len(regs)))

print('total settings', len(ranks)*len(regs))
for i, rank in enumerate(ranks):
    for j, reg in enumerate(regs):
        print('-----------------------')
        print('setting', i*len(regs)+j)
        pRRR0 = PoissonRRR(n_input=X1, n_output=Y1, act='softplus', rank=rank, loss='poisson', zeromeanXs=[0], normstdXs=[0], regList=[reg], seed=1)
        train_loss_hist, grad_hist = pRRR0.train([X_train1], Y_train1, lr=1, maxepoch=50, max_iter=20,
                                          history_size=10, grad_tol=1e-4, line_search=1,
                                          shuffle=True, track=1, patience=10, loss_tol=1e-6, progfile=None)
        test_scores, Y_test_pred = pRRR0.eval([X_val1], Y_val1, metrics=['PNLL', 'd2'])
        PNLL_cv_2fold[0, i, j] = test_scores['PNLL']
        d2_cv_2fold[0, i, j] = test_scores['d2uniform']
        
        pRRR1 = PoissonRRR(n_input=X1, n_output=Y1, act='softplus', rank=rank, loss='poisson', zeromeanXs=[0], normstdXs=[0], regList=[reg], seed=1)
        train_loss_hist, grad_hist = pRRR1.train([X_val1], Y_val1, lr=1, maxepoch=50, max_iter=20,
                                          history_size=10, grad_tol=1e-4, line_search=1,
                                          shuffle=True, track=1, patience=10, loss_tol=1e-6, progfile=None)
        test_scores, Y_test_pred = pRRR1.eval([X_train1], Y_train1, metrics=['PNLL', 'd2'])
        PNLL_cv_2fold[1, i, j] = test_scores['PNLL']
        d2_cv_2fold[1, i, j] = test_scores['d2uniform']


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)

PNLL_cv_mean = np.mean(PNLL_cv_2fold, axis=0)
rank_ix, reg_ix = np.unravel_index(np.argmax(-PNLL_cv_mean, axis=None), PNLL_cv_mean.shape)
print("Based on Poisson log-likelihood: best rank", ranks[rank_ix], "best reg", regs[reg_ix])

im = ax1.pcolormesh(-PNLL_cv_mean, edgecolors="k", linewidth=0.5)
ax1.set_aspect("equal")
divider = make_axes_locatable(ax1)
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im, cax=cax, orientation="vertical")
ax1.set_title("Poisson log-likelihood")
ax1.set_xlabel("L2 regularization")
ax1.set_xticks(np.arange(len(regs)))
ax1.set_xticklabels([f"{reg:g}" for reg in regs], rotation=60)
ax1.set_ylabel("rank")
ax1.set_yticks(np.arange(len(ranks)))
ax1.set_yticklabels(ranks)

d2_cv_mean = np.mean(d2_cv_2fold, axis=0)
rank_ix, reg_ix = np.unravel_index(np.argmax(d2_cv_mean, axis=None), d2_cv_mean.shape)
print("Based on D2 score: best rank", ranks[rank_ix], "best reg", regs[reg_ix])

im = ax2.pcolormesh(d2_cv_mean, edgecolors="k", linewidth=0.5)
ax2.set_aspect("equal")
divider = make_axes_locatable(ax2)
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im, cax=cax, orientation="vertical")
ax2.set_title(r"$D^2$ score")
ax2.set_xlabel("L2 regularization")
ax2.set_xticks(np.arange(len(regs)))
ax2.set_xticklabels([f"{reg:g}" for reg in regs], rotation=60)
ax2.set_ylabel("rank")
ax2.set_yticks(np.arange(len(ranks)))
ax2.set_yticklabels(ranks)

fig.suptitle("Cross-validation results")
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, height_ratios=[3, 1])

best_pnll_per_rank = np.max(-PNLL_cv_mean, axis=1)
best_reg_per_rank = [regs[np.argmax(-PNLL_cv_mean[i])] for i in range(len(ranks))]

ax1.plot(best_pnll_per_rank, "-o", c="green")
ax1.axhline(y=np.max(best_pnll_per_rank), linestyle="--", color="k")
ax1.axvline(x=np.argmax(best_pnll_per_rank), linestyle="--", color="k")
ax1.set_ylabel("Poisson log-likelihood")

ax2.plot(best_reg_per_rank, "-o", c="orange")
ax2.set_ylabel("L2 regularization")
ax2.set_xlabel("rank")
ax2.set_xticks(np.arange(len(ranks)))
ax2.set_xticklabels(ranks)

fig.suptitle("Cross-validation: best setting per rank")
plt.show()

Next, compare p-RRR against linear RRR on the same split.

## Compare to linear RRR

For linear RRR, each fold/reg setting is fit once to get the full-rank OLS solution, then lower ranks are obtained by truncating that solution. This is why the loop refits per regularization level but not per rank.

In [ ]:
ranks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 18, 22, 26, 30, 40, 50]
regs = [0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1000, 1e+4]

In [ ]:
mse_cv_2fold = np.zeros((2, len(ranks), len(regs)))

for l, reg in enumerate(regs):
    rrr0 = LinearRRR(bias=1, rank='max', regList=[reg], zeromeanY=0, normstdY=0, verbose=True)
    W_ols, vh = rrr0.fit([X_train1], Y_train1, [0], [0])
    for r, rank in enumerate(ranks):
        vh_r = vh[0:rank, :]
        weight = W_ols @ vh_r.T @ vh_r
        scores = rrr0.evaluate([X_val1], Y_val1, unnormY=1, weight=weight, metrics=['mse'])
        mse_cv_2fold[0, r, l] = scores['mse']
        
    rrr1 = LinearRRR(bias=1, rank='max', regList=[reg], zeromeanY=0, normstdY=0, verbose=True)
    W_ols, vh = rrr1.fit([X_val1], Y_val1, [0], [0])
    for r, rank in enumerate(ranks):
        vh_r = vh[0:rank, :]
        weight = W_ols @ vh_r.T @ vh_r
        scores = rrr1.evaluate([X_train1], Y_train1, unnormY=1, weight=weight, metrics=['mse'])
        mse_cv_2fold[1, r, l] = scores['mse']

In [ ]:
fig, ax = plt.subplots()

mse_cv_mean = np.mean(mse_cv_2fold, axis=0)
rank_ix, reg_ix = np.unravel_index(np.argmin(mse_cv_mean, axis=None), mse_cv_mean.shape)
print('Based on MSE: best rank', ranks[rank_ix], 'best reg', regs[reg_ix])

im = ax.pcolormesh(mse_cv_mean, edgecolors='k', linewidth=0.5)
ax.set_aspect('equal')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(im, cax=cax, orientation='vertical')
ax.set_title('MSE')
ax.set_xlabel('L2 regularization')
ax.set_xticks(np.arange(len(regs)))
ax.set_xticklabels(regs, rotation=60)
ax.set_ylabel('ranks')
ax.set_yticks(np.arange(len(ranks)))
ax.set_yticklabels(ranks)
fig.text(0.5, 1, 'RRR: Cross validation results', fontsize='large',ha='center', va='center')
plt.show()

LinearRRR with `bias=1` augments the design matrix with a column of ones, so the bias is absorbed into the weight matrix. The rank constraint is applied to this augmented weight; therefore, a latent rank-$R$ signal typically requires rank $R+1$ in this model.

## Refit with selected hyperparameters

Refit p-RRR and RRR on the full training set with the selected settings, then compare train/test metrics.

In [ ]:
best_prrr_rank = 10
best_prrr_reg = 1e-6

best_pRRR = PoissonRRR(
    n_input=X1,
    n_output=Y1,
    act="softplus",
    rank=best_prrr_rank,
    loss="poisson",
    zeromeanXs=[0],
    normstdXs=[0],
    regList=[best_prrr_reg],
    seed=seed,
)

In [ ]:
start = time.time()
train_loss_hist, grad_hist = best_pRRR.train(
    [X_train],
    Y_train,
    lr=1,
    maxepoch=50,
    max_iter=20,
    history_size=10,
    grad_tol=1e-4,
    line_search=1,
    shuffle=True,
    track=1,
    patience=10,
    loss_tol=1e-6,
    progfile=None,
)
elapsed_time = time.time() - start
print(f"elapsed {elapsed_time:.4f}s")

In [ ]:
train_scores_pRRR, Y_train_pred_pRRR = best_pRRR.eval(
    [X_train],
    Y_train,
    metrics=["PNLL", "cc", "r2", "d2", "MSE"],
)
test_scores_pRRR, Y_test_pred_pRRR = best_pRRR.eval(
    [X_test],
    Y_test,
    metrics=["PNLL", "cc", "r2", "d2", "MSE"],
)

In [ ]:
best_rrr_rank = 11
best_rrr_reg = 10

best_rrr = LinearRRR(
    bias=1,
    rank=best_rrr_rank,
    regList=[best_rrr_reg],
    zeromeanY=0,
    normstdY=0,
    verbose=False,
)
best_rrr.fit([X_train], Y_train, [0], [0])

In [ ]:
train_scores_rrr = best_rrr.evaluate([X_train], Y_train, metrics=["mse", "r2", "d2"])
test_scores_rrr = best_rrr.evaluate([X_test], Y_test, metrics=["mse", "r2", "d2"])

Y_train_pred_rrr = best_rrr.predict([X_train])
Y_test_pred_rrr = best_rrr.predict([X_test])

In [ ]:
# Baseline metrics from the true generating mean rates.
train_scores_true, test_scores_true = {}, {}
loss_func = nn.PoissonNLLLoss(log_input=False, full=True, reduction="mean")

true_train_mean = Y_mean[:split_idx]
true_test_mean = Y_mean[split_idx:]

train_scores_true["PNLL"] = loss_func(
    true_train_mean,
    torch.from_numpy(Y_train).float(),
).item()
test_scores_true["PNLL"] = loss_func(
    true_test_mean,
    torch.from_numpy(Y_test).float(),
).item()

train_scores_true["d2raw"] = np.array(
    [
        sklearn.metrics.d2_tweedie_score(Y_train[:, i], true_train_mean[:, i], power=1)
        for i in range(Y1)
    ]
)
test_scores_true["d2raw"] = np.array(
    [
        sklearn.metrics.d2_tweedie_score(Y_test[:, i], true_test_mean[:, i], power=1)
        for i in range(Y1)
    ]
)

train_scores_true["d2uniform"] = np.mean(train_scores_true["d2raw"])
test_scores_true["d2uniform"] = np.mean(test_scores_true["d2raw"])

train_scores_true["mse"] = sklearn.metrics.mean_squared_error(Y_train, true_train_mean)
test_scores_true["mse"] = sklearn.metrics.mean_squared_error(Y_test, true_test_mean)

train_scores_true["r2uniform"] = sklearn.metrics.r2_score(
    Y_train,
    true_train_mean,
    multioutput="uniform_average",
)
test_scores_true["r2uniform"] = sklearn.metrics.r2_score(
    Y_test,
    true_test_mean,
    multioutput="uniform_average",
)

In [ ]:
# Compute PNLL for RRR after clipping negative predictions to zero.
train_scores_rrr["PNLL"] = loss_func(
    torch.from_numpy(np.clip(Y_train_pred_rrr, a_min=0, a_max=None)).float(),
    torch.from_numpy(Y_train).float(),
).item()
test_scores_rrr["PNLL"] = loss_func(
    torch.from_numpy(np.clip(Y_test_pred_rrr, a_min=0, a_max=None)).float(),
    torch.from_numpy(Y_test).float(),
).item()

In [ ]:
heads = ["Training set", "Poisson log-likelihood", "MSE", "D2 score", "R2 score"]
train_table = [
    [
        "true mean",
        -train_scores_true["PNLL"],
        train_scores_true["mse"],
        train_scores_true["d2uniform"],
        train_scores_true["r2uniform"],
    ],
    [
        "Poisson-RRR",
        -train_scores_pRRR["PNLL"],
        train_scores_pRRR["MSE"],
        train_scores_pRRR["d2uniform"],
        train_scores_pRRR["r2uniform"],
    ],
    [
        "RRR",
        -train_scores_rrr["PNLL"],
        train_scores_rrr["mse"],
        train_scores_rrr["d2uniform"],
        train_scores_rrr["r2uniform"],
    ],
]
print(tabulate(train_table, headers=heads, tablefmt="grid"))

RRR Poisson log-likelihood and $D^2$ are computed after clipping negative RRR predictions to zero.

In [ ]:
heads = ["Test set", "Poisson log-likelihood", "MSE", "D2 score", "R2 score"]
test_table = [
    [
        "true mean",
        -test_scores_true["PNLL"],
        test_scores_true["mse"],
        test_scores_true["d2uniform"],
        test_scores_true["r2uniform"],
    ],
    [
        "Poisson-RRR",
        -test_scores_pRRR["PNLL"],
        test_scores_pRRR["MSE"],
        test_scores_pRRR["d2uniform"],
        test_scores_pRRR["r2uniform"],
    ],
    [
        "RRR",
        -test_scores_rrr["PNLL"],
        test_scores_rrr["mse"],
        test_scores_rrr["d2uniform"],
        test_scores_rrr["r2uniform"],
    ],
]
print(tabulate(test_table, headers=heads, tablefmt="grid"))

With this setup, p-RRR recovers the planted low rank and achieves strong held-out performance relative to linear RRR.

## Latent factor recovery

Compare the leading latent components from the ground-truth weight matrix, the fitted p-RRR model, and the fitted linear RRR model. We align component signs before plotting and report cosine similarity per component.

In [ ]:
# Compute SVDs of the true and fitted weight matrices.
u_true, s_true, _ = np.linalg.svd(W, full_matrices=False)

W_pRRR = (
    best_pRRR.glm.state_dict()["linear2.weight"].detach().numpy()
    @ best_pRRR.glm.state_dict()["linear1.weight"].detach().numpy()
).T
u_pRRR, s_pRRR, _ = np.linalg.svd(W_pRRR, full_matrices=False)

# Row 0 is the bias term; rows 1: are the input-to-output weight matrix.
u_rrr, s_rrr, _ = np.linalg.svd(best_rrr.weight[1:], full_matrices=False)

n_factors = min(10, u_true.shape[1], u_pRRR.shape[1], u_rrr.shape[1])

In [ ]:
# Compare aligned latent factors and record cosine similarities.
fig, axes = plt.subplots(n_factors, 1, sharex=True, figsize=(10, 10))

sim_table = [["PoissonRRR"], ["RRR"]]

for i in range(n_factors):
    cos_prrr = np.dot(u_true[:, i], u_pRRR[:, i]) / (
        np.linalg.norm(u_true[:, i]) * np.linalg.norm(u_pRRR[:, i])
    )
    sim_table[0].append(f"{abs(cos_prrr):.4f}")
    aligned_u_prrr = u_pRRR[:, i] if cos_prrr > 0 else -u_pRRR[:, i]
    axes[i].plot(
        aligned_u_prrr * s_pRRR[i],
        "-o",
        c="orange",
        markersize=4,
        label="p-RRR",
    )

    cos_rrr = np.dot(u_true[:, i], u_rrr[:, i]) / (
        np.linalg.norm(u_true[:, i]) * np.linalg.norm(u_rrr[:, i])
    )
    sim_table[1].append(f"{abs(cos_rrr):.4f}")
    aligned_u_rrr = u_rrr[:, i] if cos_rrr > 0 else -u_rrr[:, i]
    axes[i].plot(
        aligned_u_rrr * s_rrr[i],
        "-o",
        c="green",
        markersize=4,
        label="RRR",
    )

    axes[i].plot(
        u_true[:, i] * s_true[i],
        "--o",
        c="mediumpurple",
        markersize=4,
        markerfacecolor="None",
        label="true",
    )

axes[0].legend(bbox_to_anchor=(1.05, 1.0), loc="upper left")
axes[0].set_title("Latent factors")
plt.show()

In [ ]:
heads = ["cosine similarity"] + [f"PC{i + 1}" for i in range(n_factors)]
print(tabulate(sim_table, headers=heads, tablefmt="grid"))

## Test set prediction (p-RRR vs RRR vs true mean)

Visual comparison of true test-set mean rates versus predictions from p-RRR and linear RRR.

In [ ]:
true_test_mean_np = true_test_mean.numpy()

fig, axes = plt.subplots(1, 3, figsize=(7, 5))
fig.text(0.5, 1, r"test set mean $\lambda$", ha="center", va="center", fontsize="large")

vmin = min(
    np.min(true_test_mean_np),
    np.min(Y_test_pred_pRRR),
    np.min(Y_test_pred_rrr),
)
vmax = max(
    np.max(true_test_mean_np),
    np.max(Y_test_pred_pRRR),
    np.max(Y_test_pred_rrr),
)

im = axes[0].matshow(true_test_mean_np, aspect="auto", vmin=vmin, vmax=vmax)
axes[0].axis("off")
axes[0].set_title("true")

im = axes[1].matshow(Y_test_pred_pRRR, aspect="auto", vmin=vmin, vmax=vmax)
axes[1].axis("off")
axes[1].set_title("Poisson-RRR prediction")

im = axes[2].matshow(Y_test_pred_rrr, aspect="auto", vmin=vmin, vmax=vmax)
axes[2].axis("off")
axes[2].set_title("RRR prediction")

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.83, 0.15, 0.01, 0.7])
fig.colorbar(im, cax=cbar_ax)

plt.show()